# Proof-repair — Colab bootstrap (Phase 0)

Cold-start setup for the harness on a Colab T4. Done-when criterion (`plan.md` Phase 0): this notebook runs end-to-end in under 10 minutes, finishing with `pytest` green.

What it does:
1. Clones the repo (skip if you mounted it from Drive).
2. Installs `elan` and the pinned Lean toolchain from `lean-toolchain`.
3. `pip install -e .[dev]` for the Python package + test deps.
4. `pytest` against the fixture Lean files.

Mathlib is **not** warmed here — the Phase-0 fixtures (`passes.lean`, `tactic_error.lean`, `parse_error.lean`) are plain Lean with no imports. Phase 1 will add a Mathlib-restore cell that pulls a cached `.lake/` from Drive.

## 1. Clone the repo

Skip this cell if you've already mounted Google Drive and the repo is at `/content/Proof-repair`.

In [ ]:
%cd /content
![ -d Proof-repair ] || git clone https://github.com/Sfgangloff/diffusion-proof-repair.git Proof-repair
%cd /content/Proof-repair
!git rev-parse --short HEAD

## 2. Install elan + the pinned Lean toolchain

`elan` reads `lean-toolchain` and pulls the matching Lean. First run takes ~2–3 min on T4 (toolchain download).

In [ ]:
import os, subprocess, pathlib

if not pathlib.Path.home().joinpath('.elan/bin/elan').exists():
    subprocess.check_call(
        'curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh '
        '-sSf | sh -s -- -y --default-toolchain none',
        shell=True,
    )

elan_bin = str(pathlib.Path.home() / '.elan/bin')
os.environ['PATH'] = f"{elan_bin}:{os.environ['PATH']}"
!lean --version
!lake --version

## 3. Install the Python package

Editable install + dev extras (`pytest`, `pytest-xdist`).

In [ ]:
!pip install -q -e '.[dev]'

## 4. Smoke test the harness

Compiles the three fixture files directly via the CLI, then runs the full test suite.

In [ ]:
!python -m proofrepair compile tests/fixtures/lean/passes.lean
!python -m proofrepair compile tests/fixtures/lean/tactic_error.lean || true
!python -m proofrepair compile tests/fixtures/lean/parse_error.lean || true

In [ ]:
!pytest -q

## 5. (Phase 1 placeholder) Restore Mathlib cache from Drive

Once Phase 1 lands and benchmark files import Mathlib, this cell will:
- mount Drive,
- copy a pre-built `.lake/` over from `Drive/proofrepair-cache/`,
- run `lake build` to verify the cache is usable.

Left as a stub for now — the Phase-0 fixtures don't import Mathlib.